# Implementing the RAG flow

Now that we understand the RAG flow conceptually, let's implement it step by step. We'll walk through a complete example that demonstrates how to chunk text, generate embeddings, store them in a vector database, and perform similarity searches.

### The Five-Step RAG Implementation
Our implementation follows the same five steps we discussed previously:

1. Chunk the text by section
2. Generate embeddings for each chunk
3. Create a vector store and add each embedding to it
4. Generate an embedding for the user's question
5. Search the store to find the most relevant chunks

### Step 1: Chunking the Text
First, we load our document and split it into manageable sections:

In [1]:
# Add repo root to path
import sys
sys.path.append("../..")

In [6]:
from src.rag import chunk_by_section, generate_embedding, VectorIndex

In [3]:
with open("report.md", "r") as f:
    text = f.read()

In [4]:
chunks = chunk_by_section(text)
chunks[2]  # Test to see the table of contents

'Table of Contents\n\n1.  Executive Summary\n2.  Table of Contents\n3.  Methodology\n4.  Section 1: Medical Research - Understanding XDR-471 Syndrome\n5.  Section 2: Software Engineering - Project Phoenix Stability Enhancements\n6.  Section 3: Financial Analysis - Q3 Performance and Outlook\n7.  Section 4: Scientific Experimentation - Characterization of Material Composite XT-5\n8.  Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts\n9.  Section 6: Product Engineering - Finalizing Model Zircon-5 Specifications\n10. Section 7: Historical Research - Re-evaluating the Galveston Accords (1921)\n11. Section 8: Project Management - Progress on Project Cerberus Phase 2B\n12. Section 9: Pharmaceutical Development - Compound CTX-204b Phase IIa Update\n13. Section 10: Cybersecurity Analysis - Incident Response Report\n14. Future Directions\n'

### Step 2: Generate Embeddings
Next, we create embeddings for all our chunks at once:

In [5]:
embeddings = generate_embedding(chunks)

### Step 3: Store in Vector Database

Now we create our vector store and populate it with embeddings and their associated text:

In [7]:
store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

Notice that we store both the embedding and the original text content. This is crucial because when we search later, we need to return the actual text, not just the numerical embedding values.

#### Why Store the Original Text?
When we query our vector database, getting back just the embedding numbers isn't useful. We need the actual text that was used to generate those embeddings. That's why we include the original chunk text (or at least a reference to it) alongside each embedding in our database.

### Step 4: Process User Queries
When a user asks a question, we generate an embedding for their query:



In [14]:
user_embedding = generate_embedding("What happened with INC-2023-Q4-011?")

### Step 5: Find Relevant Content
Finally, we search our vector store to find the most similar chunks:

In [16]:
results = store.search(user_embedding, 3)

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200], "\n")

0.48317230118293863 
 Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2 

0.509342448667044 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems 

0.5661010116719827 
 Table of Contents

1.  Executive Summary
2.  Table of Contents
3.  Methodology
4.  Section 1: Medical Research - Understanding XDR-471 Syndrome
5.  Section 2: Software Engineering - Project Phoenix St 



The search results show us which sections of our document are most relevant to the user's question, along with similarity scores.

#### Understanding the Results
When we run our example query about the software engineering department, we get back:

- "Section 2: Software Engineering" with a distance of 0.484 (closest match)
- "This year's cross-domain insights" with a distance of 0.487 (second closest)

Lower distance values indicate higher similarity, so Section 2 is the most relevant to our query.

### What's Next?
This implementation works well for basic cases, but there are scenarios where it doesn't perform as expected. In the next sections, we'll explore improvements to make our RAG system more robust and accurate.

The key takeaway is that RAG is fundamentally about converting text to numbers (embeddings), storing those numbers efficiently, and then using mathematical similarity to find relevant content when users ask questions.